# Wyscout API v3 and v4 with kloppy

This notebook downloads the events of one match from the Wyscout API, from **v3** and from **v4**. Then it loads both files with `kloppy.wyscout.load()` and compares the results.

v3 and v4 events have the same structure. Only the `formations` object (included with `fetch=formations`) has a new shape in v4. See [PySport/kloppy#652](https://github.com/PySport/kloppy/issues/652).

## Setup

You need Wyscout API credentials. Set `CLIENT_ID` and `CLIENT_SECRET` as environment variables, or put them in `notebooks/.env`:

```
CLIENT_ID=...
CLIENT_SECRET=...
```

`.env` and the downloaded data in `notebooks/data/` are git-ignored. **Do not commit Wyscout data or this notebook with its outputs**: the data is licensed.

In [ ]:
import base64
import json
import os
from pathlib import Path
import time
from typing import Optional

import requests

from kloppy import wyscout
from kloppy.domain import EventType

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)


def load_env_file(path: Path) -> None:
    """Read KEY=VALUE lines into os.environ. Existing variables are kept."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_env_file(Path(os.getenv("WYSCOUT_ENV_FILE", NOTEBOOK_DIR / ".env")))
assert os.getenv("CLIENT_ID") and os.getenv("CLIENT_SECRET"), (
    "Set CLIENT_ID and CLIENT_SECRET"
)

## A small API client

The Wyscout API uses Basic authentication. The base URL contains the version: `https://apirest.wyscout.com/v3/` or `https://apirest.wyscout.com/v4/`.

Sometimes the API answers `"Data temporary unavailable"`, or `429` when there are more than 12 requests per second. The client tries again a few times when this happens.

In [ ]:
BASE_URL = "https://apirest.wyscout.com/{version}/"


def get(
    version: str, path: str, params: Optional[dict] = None, retries: int = 5
) -> dict:
    token = base64.b64encode(
        f"{os.environ['CLIENT_ID']}:{os.environ['CLIENT_SECRET']}".encode()
    ).decode()
    for attempt in range(1, retries + 1):
        response = requests.get(
            BASE_URL.format(version=version) + path,
            params=params or {},
            headers={"Authorization": f"Basic {token}"},
            timeout=60,
        )
        temporary = (
            response.status_code in (429, 500, 502, 503, 504)
            or "temporary unavailable" in response.text
        )
        if not temporary or attempt == retries:
            break
        time.sleep(2 * attempt)
    if not response.ok:
        # Show the API's own message, e.g. "Data temporary unavailable"
        raise requests.HTTPError(
            f"{response.status_code} after {attempt} attempt(s) for "
            f"{response.url}: {response.text[:500]}",
            response=response,
        )
    return response.json()

## Choose a match

Set `MATCH_ID` to load a match that you know. If it is `None`, the notebook takes the most recent match of `TEAM_ID` in `SEASON_ID` (Southend United, National League 2025/26).

In [ ]:
TEAM_ID = 1687  # Southend United
SEASON_ID = 191757  # National League 2025/26
MATCH_ID = None

if MATCH_ID is None:
    matches = get("v4", f"teams/{TEAM_ID}/matches", {"seasonId": SEASON_ID})[
        "matches"
    ]
    played = [m for m in matches if m["status"] == "Played"]
    latest = max(played, key=lambda m: m["dateutc"])
    MATCH_ID = latest["matchId"]
    print(latest["date"], latest.get("label", ""))

MATCH_ID

HTTPError: 400 after 1 attempt(s) for https://apirest.wyscout.com/v4/teams/1687/matches?seasonId=191757: { "error": { "code": 403, "message": "Forbidden: You don't have permission to access this resource" } }

## Download the events from v3 and v4

kloppy needs the teams, players, match details and formations together with the events, so we ask for them with `fetch`. We save each response to `data/`, so that we only download it one time.

In [ ]:
FETCH = "teams,players,match,coaches,formations,substitutions"


def download_events(version: str, match_id: int) -> Path:
    path = DATA_DIR / f"wyscout_events_{version}_{match_id}.json"
    if not path.exists():
        data = get(version, f"matches/{match_id}/events", {"fetch": FETCH})
        path.write_text(json.dumps(data))
    return path


v3_path = download_events("v3", MATCH_ID)
v4_path = download_events("v4", MATCH_ID)
v3_path.name, v4_path.name

## What is different in the raw data

The events have the same keys. The `formations` object is different:

- v3 groups formations by team ID, then period, then start second, then scheme.
- v4 has a `home` list and an `away` list. One formation can last for more than one period (`matchPeriodStart` / `matchPeriodEnd`).

In [ ]:
raw_v3 = json.loads(v3_path.read_text())
raw_v4 = json.loads(v4_path.read_text())

print("top-level keys v3:", sorted(raw_v3))
print("top-level keys v4:", sorted(raw_v4))
print(
    "same event keys:  ", set(raw_v3["events"][0]) == set(raw_v4["events"][0])
)
print()
print("v3 formations keys:", list(raw_v3["formations"]))
print("v4 formations keys:", list(raw_v4["formations"]))

In [ ]:
first_v4_formation = raw_v4["formations"]["home"][0]
{k: v for k, v in first_v4_formation.items() if k not in ("players", "team")}

## Load both files with kloppy

`wyscout.load()` detects the version from the data. You can also set it with `data_version`.

In [ ]:
from kloppy._providers.wyscout import identify_deserializer

print("v3 file detected as:", identify_deserializer(v3_path).__name__)
print("v4 file detected as:", identify_deserializer(v4_path).__name__)

dataset_v3 = wyscout.load(event_data=v3_path, data_version="V3")
dataset_v4 = wyscout.load(event_data=v4_path, data_version="V4")

In [ ]:
for name, dataset in [("v3", dataset_v3), ("v4", dataset_v4)]:
    home, away = dataset.metadata.teams
    print(
        f"{name}: {home.name} ({home.starting_formation.value}) vs "
        f"{away.name} ({away.starting_formation.value}), "
        f"{len(dataset.events)} events, periods {[p.id for p in dataset.metadata.periods]}"
    )

### Starting line-ups

The starting players and positions come from the formations. This is the part that is different in v4, so the two tables must be the same.

In [ ]:
import pandas as pd


def starting_xi(dataset):
    return pd.DataFrame(
        [
            {
                "team": team.name,
                "player_id": player.player_id,
                "player": player.full_name,
                "position": player.starting_position.name,
            }
            for team in dataset.metadata.teams
            for player in team.players
            if player.starting
        ]
    ).set_index(["team", "player_id"])


xi_v3, xi_v4 = starting_xi(dataset_v3), starting_xi(dataset_v4)
print(
    "starting XIs are the same:", xi_v3.sort_index().equals(xi_v4.sort_index())
)
xi_v4

### Events

Compare the events of the two datasets. We leave out the synthetic `FORMATION_CHANGE` events that kloppy creates.

In [ ]:
def events_df(dataset):
    return (
        dataset.filter(lambda e: e.event_type != EventType.FORMATION_CHANGE)
        .to_df(
            "event_id",
            "event_type",
            "period_id",
            "timestamp",
            "player_id",
            "coordinates_*",
            "result",
        )
        .set_index("event_id")
        .sort_index()
    )


df_v3, df_v4 = events_df(dataset_v3), events_df(dataset_v4)
print("same number of events:", len(df_v3) == len(df_v4))
print("same events:          ", df_v3.equals(df_v4))
df_v4.head()

In [ ]:
df_v4["event_type"].value_counts()

## Why v4 needs its own deserializer

The v3 deserializer reads the v3 formations shape, so it fails on v4 data.

In [ ]:
try:
    wyscout.load(event_data=v4_path, data_version="V3")
except Exception as e:
    print(f"{type(e).__name__}: {e!r}")